Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

    A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
    A function or coroutine to execute.


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:openai/gpt-oss-120b")

In [5]:
from langchain.tools import tool

@tool
def get_current_weather(location: str) -> str:
    """Get the current weather in a given location"""
    return f"The current weather in {location} is sunny with a temperature of 25°C." 

model_with_tools = model.bind_tools([get_current_weather])   

In [6]:
response = model_with_tools.invoke("What is the current weather in New York?")
print(response)

content='' additional_kwargs={'reasoning_content': 'The user asks for current weather in New York. We have a function get_current_weather. Use it.', 'tool_calls': [{'id': 'fc_32297312-176f-4348-8771-3288d827674a', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_current_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 132, 'total_tokens': 183, 'completion_time': 0.105891744, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.005292035, 'prompt_tokens_details': None, 'queue_time': 0.212268025, 'total_time': 0.111183779}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4a7bb72c24', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a13a47-0b62-7b71-a3ff-a46729b073d0-0' tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'New York'}, 'id': 'fc_32297312-176f-4348-8771-3288d827674a', 'type': 'tool_ca

In [8]:
for tool_call in response.tool_calls:
    print(f"Tool called : {tool_call['name']}")
    print(f"Tool input : {tool_call['args']}")

Tool called : get_current_weather
Tool input : {'location': 'New York'}


In [10]:
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
for tool_call in ai_msg.tool_calls:
    tool_result = get_current_weather.invoke(tool_call)
    messages.append(tool_result)
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The current weather in Boston is sunny with a temperature of 25 °C.


In [11]:


messages



[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "What\'s the weather in Boston?" Need to get current weather using provided function get_current_weather. We\'ll call function with location "Boston".', 'tool_calls': [{'id': 'fc_b9fd3e1a-9cb2-45b4-91d8-cfcdfca13999', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_current_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 129, 'total_tokens': 188, 'completion_time': 0.121982052, 'completion_tokens_details': {'reasoning_tokens': 31}, 'prompt_time': 0.006165669, 'prompt_tokens_details': None, 'queue_time': 0.214065179, 'total_time': 0.128147721}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a655021f7d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a13a4d-1679-76d3-9cec-c4d0a2aa25cb-0', 